# ERDOS-STRAUS SIEVE — Google Colab Gemini LLM Verifier
### Lead R&D: DaShawn (African American Developer & Mathematician)

This notebook is part of the **Substrate Delta Sieve** and **Ghost Braid** telemetry systems. It is designed to:
1. Mount Google Drive to read Erdős-Straus sieve checkpoints (`KAGGLE_OUTPUT_RECORD.jsonl`).
2. Perform **exact integer arithmetic validation** on the discovered fraction triples.
3. Use the **Gemini LLM** to analyze the mathematical density and modular symmetries of the hot corridor ($n \equiv 0 \pmod{24}$).

In [ ]:
# 1. MOUNT GOOGLE DRIVE
from google.colab import drive
import os, sys
from pathlib import Path

try:
    drive.mount('/content/drive')
    DRIVE_DIR = Path("/content/drive/MyDrive/erdos-straus-solver")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"[GHOST BRAID] Mounted Google Drive. Target output directory: {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = Path(".")
    print(f"[WARNING] Could not mount Google Drive. Running in local Colab space: {e}")

In [ ]:
# 2. GENERATIVE AI INITIALIZATION
# Make sure to add your GEMINI_API_KEY to the Colab Secrets (key icon on the left panel)
import google.generativeai as genai
from google.colab import userdata

try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=gemini_key)
    # Initialize Gemini 1.5 Pro for complex mathematical reasoning
    model = genai.GenerativeModel('gemini-1.5-pro')
    print("\u2713 Successfully initialized Gemini API client.")
except Exception as e:
    print(f"[WARNING] Gemini API key not found in Colab Secrets: {e}")
    print("Please add your 'GEMINI_API_KEY' in the Secrets tab to enable LLM verification.")

In [ ]:
# 3. DETERMINISTIC EXACT MATH VERIFIER
import json

JSONL_OUTPUT = DRIVE_DIR / "KAGGLE_OUTPUT_RECORD.jsonl"
STATE_FILE = DRIVE_DIR / "erdos_output.json"

def verify_egyptian_fraction(n, x, y, z):
    """Exact integer math check for 4/n = 1/x + 1/y + 1/z to avoid floating point errors."""
    lhs = 4 * x * y * z
    rhs = n * (y * z + x * z + x * y)
    return lhs == rhs

records = []
if JSONL_OUTPUT.exists():
    with open(JSONL_OUTPUT, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    print(f"Loaded {len(records):,} records from {JSONL_OUTPUT.name}")
    
    # Verify a subset of recent records
    verification_failures = []
    to_check = records[-100:] if len(records) >= 100 else records
    for r in to_check:
        n = r.get("n")
        triple = r.get("triple")
        if triple and len(triple) == 3:
            x, y, z = triple
            if not verify_egyptian_fraction(n, x, y, z):
                verification_failures.append((n, triple))
                
    if verification_failures:
        print(f"\u274c Math verification failed for {len(verification_failures)} records!")
        for f_n, f_t in verification_failures[:5]:
            print(f"  Fail: n={f_n}, triple={f_t}")
    else:
        print(f"\u2713 Math verification passed for all {len(to_check)} sampled records!")
else:
    print(f"No records found at {JSONL_OUTPUT}. Please run the sieve first or copy the dataset.")

In [ ]:
# 4. LLM SUBSTRATE ANALYSIS & REPORT GENERATION
from datetime import datetime

if records:
    # Sample records for prompt context
    sample_records = records[-10:] if len(records) >= 10 else records
    
    prompt = (
        "You are the mathematical reasoning agent of the Substrate Delta Sieve Engine.\n"
        "Analyze these latest Erd\u0151s-Straus sieve solutions for multiples of 24 (the hot corridor):\n\n"
    )
    for r in sample_records:
        n = r.get("n")
        triple = r.get("triple")
        sols_count = r.get("num_solutions")
        depth = r.get("depth")
        prompt += f"- n={n}: triple={triple}, solutions_found={sols_count}, classification={depth}\n"
        
    prompt += (
        "\nVerify that the first solution in each n holds mathematically: 4/n = 1/x + 1/y + 1/z.\n"
        "Then, explain why multiples of 24 (mod24=0) always yield at least 2 distinct solutions (e.g. Identity 1 and Identity 2),\n"
        "and describe how the divisor search finds additional solutions.\n"
        "Provide a concise mathematical analysis of the solution density and modular symmetry.\n"
        "Sign the report as: Substrate Delta Sieve Agent (Under direction of Lead R&D DaShawn)."
    )
    
    print("Requesting mathematical analysis from Gemini...")
    try:
        response = model.generate_content(prompt)
        analysis_text = response.text
        print("\n=== GEMINI MATHEMATICAL RESONANCE ANALYSIS ===\n")
        print(analysis_text)
        
        # Save report to Google Drive
        report_path = DRIVE_DIR / "resonance_report.md"
        report_path.write_text(
            f"# Erd\u0151s-Straus Resonance Report\n"
            f"**Generated:** {datetime.now().isoformat()}\n"
            f"**Lead R&D:** DaShawn (African American Developer & Mathematician)\n\n"
            f"{analysis_text}\n"
        )
        print(f"\n\u2713 Saved report to Google Drive: {report_path}")
    except Exception as e:
        print(f"Gemini API request failed: {e}")
else:
    print("Cannot perform LLM analysis: No records loaded.")